# Find NAC imagery overlapping the WAC poster AOI

Scan the two processed NAC collections, using **10 worker processes**, to choose
an image for [crater labeling](crater_labeling.ipynb) and then
[chip creation](chip_example.ipynb). Run the cells in order with `lfm_kernel`.

**Coverage (%) = 100 × area(NAC footprint ∩ WAC AOI) / area(WAC AOI).**
The denominator is the requested WAC AOI, not the full WAC scene or the NAC scene.
Areas use a lunar Lambert azimuthal equal-area projection centered on the AOI;
the Moon's radius comes from the repository's IAU:30100 definition.

Only raster metadata is read. The footprint follows the affine outer pixel
boundary, including rotation, with 64 segments per edge before reprojection.
**NoData margins are included:** this is a candidate search, not a valid-pixel
coverage measurement. Inspect the selected NAC image before labeling it.
Shared source files and indexes are read-only.

## 1. Imports and repository discovery

Run from the repository root or its top-level `notebooks/` directory.
The worker is an importable Python function so multiprocessing with `spawn`
works in Jupyter. Dependencies are already in `requirements_container.txt`:
NumPy, Rasterio, PyProj, and Shapely. No training libraries are needed.

In [ ]:
from concurrent.futures import ProcessPoolExecutor
from datetime import datetime, timezone
from functools import partial
import json
import multiprocessing
from pathlib import Path
import sys

working_dir = Path(str(Path.cwd()).replace(
    "/panfs/ccds02/nobackup", "/explore/nobackup"
))
repo_root = next(
    (p for p in (working_dir, working_dir.parent)
     if (p / "lfm/data_processing").is_dir() and (p / "TMS").is_dir()),
    None,
)
if repo_root is None:
    raise FileNotFoundError("Run from the repository root or top-level notebooks/ directory.")
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from lfm.data_processing.raster_coverage import (
    coverage_context,
    inspect_raster_coverage,
)

print(f"Repository: {repo_root}")

## 2. User configuration

`AOI_NWSE` is **north latitude, west longitude, south latitude, east longitude**
in lunar IAU:30100 degrees. This search supports ordinary nonpolar AOIs that do
not cross the antimeridian. The defaults below are your WAC AOI.

`RASTER_GLOB` searches immediately inside each directory, matching your `*.tif`
request. Change it to `**/*.tif` only if you also want nested directories.
Use a session/allocation with at least 10 CPUs for `MAX_WORKERS = 10`.

In [ ]:
AOI_NWSE = (-20.9333825982, 147.9653785664, -21.8923820897, 148.9955331735)
NAC_DATA_DIRS = [
    Path("/explore/nobackup/projects/lfm/processed_data/Lunar/LRO_NAC_IMP"),
    Path("/explore/nobackup/projects/lfm/processed_data/Lunar/LRO_NAC_Pho_Sites"),
]
RASTER_GLOB = "*.tif"
MAX_WORKERS = 10
EDGE_SEGMENTS = 64
OUTPUT_BASE_DIR = repo_root / "notebooks/outputs/nac_aoi_overlap"

## 3. Discover source files and validate the AOI

Full paths are dictionary keys so identical basenames in different collections
cannot overwrite each other. Missing directories stop the scan rather than
silently returning an incomplete inventory. An empty collection is reported.

In [ ]:
LUNAR_WKT_PATH = repo_root / "TMS/IAU_30100_2015.wkt"
lunar_wkt = LUNAR_WKT_PATH.read_text(encoding="utf-8")
_, _, _, aoi_area_m2, _ = coverage_context(
    tuple(AOI_NWSE), lunar_wkt, EDGE_SEGMENTS
)
if not isinstance(MAX_WORKERS, int) or MAX_WORKERS < 1:
    raise ValueError("MAX_WORKERS must be a positive integer.")

missing_dirs = [str(path) for path in NAC_DATA_DIRS if not path.is_dir()]
if missing_dirs:
    raise FileNotFoundError("NAC directories unavailable: " + ", ".join(missing_dirs))

source_files = set()
for directory in NAC_DATA_DIRS:
    found = sorted(str(path.absolute()) for path in directory.glob(RASTER_GLOB) if path.is_file())
    print(f"{directory}: {len(found):,} matching TIFFs")
    if not found:
        print("  WARNING: no files matched in this collection.")
    source_files.update(found)
source_files = sorted(source_files)
if not source_files:
    raise FileNotFoundError("No TIFFs matched. Check NAC_DATA_DIRS and RASTER_GLOB.")

print(f"Unique files: {len(source_files):,}")
print(f"WAC AOI area: {aoi_area_m2 / 1_000_000:,.3f} km²")

## 4. Scan in parallel

Each worker opens and closes its own raster handle. Results and dictionaries
are collected only by the notebook process. Errors are retained separately and
are never converted to zero coverage. A zero means a successful footprint
inspection with no area overlap.

- `overlap_by_filename`: requested `{full_filename: overlap_int}` dictionary,
  rounded to the nearest whole percentage point.
- `overlap_percent_by_filename`: precise percentages used for ranking; a small
  positive overlap can round to zero in the integer dictionary.
- `scan_errors`: `{full_filename: error_message}` for failed inspections.

Rerunning this cell resets the dictionaries for a fresh scan.

In [ ]:
overlap_by_filename = {}
overlap_percent_by_filename = {}
scan_errors = {}
scan_started_utc = datetime.now(timezone.utc).isoformat()
worker = partial(
    inspect_raster_coverage,
    aoi_nwse=tuple(AOI_NWSE),
    lunar_wkt=lunar_wkt,
    edge_segments=EDGE_SEGMENTS,
)

with ProcessPoolExecutor(
    max_workers=MAX_WORKERS,
    mp_context=multiprocessing.get_context("spawn"),
) as pool:
    results = pool.map(worker, source_files, chunksize=8)
    for completed, result in enumerate(results, start=1):
        filename = result["filename"]
        if result["error"] is not None:
            scan_errors[filename] = result["error"]
        else:
            percent = result["overlap_percent"]
            overlap_percent_by_filename[filename] = percent
            overlap_by_filename[filename] = int(round(percent))
        if completed == 1 or completed % 100 == 0 or completed == len(source_files):
            print(f"Checked {completed:,}/{len(source_files):,} files; errors: {len(scan_errors):,}", flush=True)

scan_finished_utc = datetime.now(timezone.utc).isoformat()
print(f"Inspected: {len(overlap_by_filename):,}; failed: {len(scan_errors):,}")
print(f"Positive overlap: {sum(p > 0 for p in overlap_percent_by_filename.values()):,}")
if scan_errors:
    print("First five errors (all errors are saved below):")
    for filename, error in list(scan_errors.items())[:5]:
        print(f"  {filename}: {error}")

## 5. Save results and print the top five overlapping files

Every run gets its own directory. `overlap_by_filename.json` is the simple
integer dictionary; `scan_report.json` preserves precise percentages, errors,
AOI/CRS, source directories, and the coverage definition. Ranking uses precise
percentages, with full filename as the tie-breaker. Only positive overlaps are
listed; fewer than five candidates is a valid result.

In [ ]:
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
output_dir = OUTPUT_BASE_DIR / run_id
output_dir.mkdir(parents=True, exist_ok=False)
(output_dir / "overlap_by_filename.json").write_text(
    json.dumps(overlap_by_filename, indent=2, sort_keys=True) + "\n", encoding="utf-8"
)
report = {
    "aoi_nwse_iau30100": list(AOI_NWSE),
    "aoi_area_m2": aoi_area_m2,
    "lunar_geographic_wkt": lunar_wkt,
    "coverage_definition": "100 * area(raster affine footprint intersect WAC AOI) / area(WAC AOI)",
    "area_method": "AOI-centered lunar Lambert azimuthal equal-area; densified boundaries",
    "includes_nodata_margins": True,
    "edge_segments": EDGE_SEGMENTS,
    "source_directories": [str(path) for path in NAC_DATA_DIRS],
    "raster_glob": RASTER_GLOB,
    "worker_count": MAX_WORKERS,
    "scan_started_utc": scan_started_utc,
    "scan_finished_utc": scan_finished_utc,
    "discovered_file_count": len(source_files),
    "overlap_percent_by_filename": overlap_percent_by_filename,
    "errors": scan_errors,
}
(output_dir / "scan_report.json").write_text(
    json.dumps(report, indent=2, sort_keys=True, allow_nan=False) + "\n", encoding="utf-8"
)
print(f"Saved scan to: {output_dir}")

top_five = sorted(
    ((filename, percent) for filename, percent in overlap_percent_by_filename.items() if percent > 0),
    key=lambda item: (-item[1], item[0]),
)[:5]
print("\nTop NAC files by percentage of WAC AOI covered:")
for rank, (filename, percent) in enumerate(top_five, start=1):
    print(f"{rank}. {percent:.6f}%  {filename}")
if not top_five:
    print("No positive overlaps found. Check scan_errors before concluding there is no coverage.")

## 6. Use a candidate for the poster

1. Choose a path from `top_five`, inspect the imagery and its NoData margins,
   and set `DEFAULT_RASTER` in [crater_labeling.ipynb](crater_labeling.ipynb).
   Save the finished crater annotations as a GeoPackage.
2. In [chip_example.ipynb](chip_example.ipynb), set `MODALITY = "nac"`,
   `SOURCE_RASTER` to that original NAC raster, and `LABEL_PATH` to its finished
   annotations. Set `AOI_NWSE` to the geographic region you want to illustrate.
3. Run the chip notebook with WAC and NAC separately, using the **same AOI**,
   to generate the inspection figures. The two runs keep their source-native
   resolutions; matching AOIs do not imply matching pixel dimensions.

This WAC search AOI is roughly 30 km across. At 1 m resolution, a single NAC
chip spanning it would contain hundreds of millions of pixels, before adding
static bands. For poster figures, choose a smaller common AOI inside the
selected NAC image's **valid** coverage, and use that exact AOI for both runs.
An unlabeled region outside a partially overlapping NAC scene should not be
interpreted as confirmed crater-free ground.